# Getting Started with gprMax

This interactive tutorial demonstrates how to use `gprMax` to simulate Ground Penetrating Radar (GPR). We will:

1.  Define a simple 2D model of a metal cylinder buried in a dielectric half-space.
2.  Run the simulation using the `gprMax` Python API.
3.  Visualize the results (A-scan) using `matplotlib`.

## prerequisites

Ensure `gprMax` is installed and the environment is active.

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from gprMax.gprMax import api

## 1. Defining the Model

gprMax uses input files (`.in`) to define the simulation parameters, geometry, and materials. Here we define a standard example: a metal cylinder buried in a half-space.

We will write this configuration to a file named `my_cylinder_model.in`.

In [ ]:
input_file_name = 'my_cylinder_model.in'

input_file_content = """
#title: A-scan from a metal cylinder buried in a dielectric half-space
#domain: 0.240 0.210 0.002
#dx_dy_dz: 0.002 0.002 0.002
#time_window: 3e-9

#material: 6 0 1 0 half_space

#waveform: ricker 1 1.5e9 my_ricker
#hertzian_dipole: z 0.100 0.170 0 my_ricker
#rx: 0.140 0.170 0

#box: 0 0 0 0.240 0.170 0.002 half_space
#cylinder: 0.120 0.080 0 0.120 0.080 0.002 0.010 pec

#geometry_view: 0 0 0 0.240 0.210 0.002 0.002 0.002 0.002 cylinder_half_space n
"""

with open(input_file_name, 'w') as f:
    f.write(input_file_content.strip())

print(f"Created input file: {input_file_name}")

## 2. Running the Simulation

We can run the simulation directly from Python using `gprMax.gprMax.api`. This is equivalent to running `python -m gprMax my_cylinder_model.in` from the command line.

In [ ]:
# Run the simulation
# n=1: Run a single simulation
# geometry_only=False: Run the full electromagnetic simulation
api(input_file_name, n=1, geometry_only=False)

## 3. Visualizing Results

gprMax produces HDF5 output files (`.out`). We can read these using `h5py` and plot the field components. Here we plot the `Ez` component (Electric field in the Z direction) recorded at the receiver.

In [ ]:
# Define the output file name (gprMax appends .out)
output_file_name = 'my_cylinder_model.out'

if os.path.exists(output_file_name):
    # Read the output file
    f = h5py.File(output_file_name, 'r')

    # Get the electric field Ez component from the receiver (rx1)
    # The path in HDF5 is usually /rxs/rx1/Ez
    ez_data = f['rxs']['rx1']['Ez'][: ]
    dt = f.attrs['dt']
    iterations = f.attrs['Iterations']

    # Create time array
    time = np.linspace(0, (iterations - 1) * dt, iterations)

    # Plotting
    plt.figure(figsize=(10, 6))
    plt.plot(time * 1e9, ez_data)
    plt.xlabel('Time (ns)')
    plt.ylabel('Electric Field Ez (V/m)')
    plt.title('A-scan of a metal cylinder')
    plt.grid(True)
    plt.show()
    
    f.close()
else:
    print("Output file not found!")